# Controllability V2 Summary Analysis

This notebook compares the arm-aware `v2` controllability outputs across studies and models.


In [ ]:
import sys
from pathlib import Path

for candidate_root in [Path.cwd(), Path.cwd().parent]:
    src_dir = candidate_root / "src"
    if src_dir.exists() and str(src_dir.resolve()) not in sys.path:
        sys.path.insert(0, str(src_dir.resolve()))

import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

RESULTS_DIR = next(
    (p for p in [Path("results"), Path("../results"), Path("../../results")] if p.exists()),
    Path("results"),
)
print(f"Using RESULTS_DIR: {RESULTS_DIR.resolve()}")


In [ ]:
def load_controllability_v2_summary_rows():
    study_rows = []
    if not RESULTS_DIR.exists():
        return pd.DataFrame()

    for model_dir in sorted(path for path in RESULTS_DIR.iterdir() if path.is_dir()):
        summary_path = model_dir / "controllability_v2_summary.json"
        if not summary_path.exists():
            continue
        payload = json.loads(summary_path.read_text(encoding="utf-8"))
        for study_key, study_payload in sorted((payload.get("studies") or {}).items()):
            for arm_name, arm_payload in sorted((study_payload.get("arms") or {}).items()):
                primary_metric = arm_payload.get("primary_metric") or {}
                study_rows.append(
                    {
                        "model": payload.get("model", model_dir.name),
                        "study": study_key,
                        "arm": arm_name,
                        "primary_metric_name": primary_metric.get("metric_name"),
                        "primary_metric_value": primary_metric.get("value"),
                    }
                )

    return pd.DataFrame(study_rows)


In [ ]:
summary_df = load_controllability_v2_summary_rows()
if summary_df.empty:
    print("No controllability v2 summary files found yet.")
else:
    display(summary_df.sort_values(["study", "arm", "primary_metric_value"], ascending=[True, True, False]).reset_index(drop=True))


## Primary Metrics Across Studies and Arms


In [ ]:
if summary_df.empty:
    print("Skipping summary plot - no v2 summary rows found.")
else:
    fig, ax = plt.subplots(figsize=(18, 8))
    sns.barplot(data=summary_df, x="study", y="primary_metric_value", hue="arm", ax=ax)
    ax.set_title("Controllability v2 primary metrics by study and arm", fontsize=14, fontweight="bold")
    ax.set_xlabel("Study")
    ax.set_ylabel("Primary metric value")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()
